<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


# Your Places, Week 4 Session 1: Filtering Your Places

### CP101: Introduction to Urban Data Analytics
#### Monday, Python Basics II

**Which places would you recommend to a classmate?** 

In Lab 1.2 you built your `myplaces` CSV, one row per place. Today Python reads that file back. Section 1 opens it with the standard library and picks values out of it with lists and indexing. Then you rate your places, and use the concepts we covered today. namely, truthiness, conditionals, and loops, to sort the places based on what you would highly recommend.

Work with a partner. Predict the output before running each example, then take turns writing code.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 1. Read your places file

In **Lab 1.2**, you created a CSV file called something like:

```text
firstname_studentid_myplaces.csv
```

Today, Python will read that file.

### Find your file

Your CSV is in your **lab folder**, while this notebook is in a different folder. We need a **relative path** from this notebook to your file.

First, in a new cell, run:

```python
pwd
```

`pwd` means **print working directory**. It shows where this notebook is currently running.

Then, in the DataHub file browser:

1. Open `materials-fa26/lab/w02/`.
2. Find your myplaces `.csv` file.
3. Right click it and choose **Copy Path**.
4. createa variable called `path`, referencing what you just copied and in the copied path, replace `materials-fa26` with `../../..`.

For example:

```text
path = materials-fa26/lab/w02/alex_1234567_myplaces.csv
```

becomes:

```text
path = ../../../lab/w02/alex_1234567_myplaces.csv
```

Each `..` means **go up one folder**. Here, `../../..` takes you from the notebook's folder back to `materials-fa26`.

In the next cell, replace the placeholder filename with your own and check that the path works.


### ✏️ Try it out! 
Modify the code below to point to your own csv file, and run it. It should print `True` if the path is correct.

In [ ]:

import os  # import the `os` module to work with file paths

path = "../../../lab/w02/firstname_studentid_myplaces.csv"

os.path.exists(path) # Check if the file exists at the specified path

### Reading the file

`open(path)` opens the file. We use it together with `with`, which closes the file again automatically when the indented block ends. This `with ... as` pattern in Python is called a **context manager**.

```python
with open(path) as f:
    lines = f.readlines()
```

`f.readlines()` reads the file line by line and puts every line, as a string, into a list called `lines`.

The first line of your CSV is the header, so your first recommended place is at position `1`.

In [ ]:
with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines() # note that we have an indented block here with the `with` statement
print(lines[1])   # the first place, one long string
print('-'*20)
print(type(lines))  # the type of the variable `lines` is a list
print('-'*20)
print(type(lines[1]))  # the type of the variable `lines[1]` is a string

### Splitting a line into values

One long string is not very useful. We want to access the name, the latitude, the category, each on its own.

The obvious idea is to cut the string at every comma with `.split(",")`. Try it on the example row below. It has the same format as your file, so everyone sees the same thing.

In [ ]:
example = 'Doe Library,"Doe Library, Campanile Way, Berkeley, CA 94720",37.8724,-122.2592,reading,"Quiet north reading room, best for long sessions",,'
parts = example.split(",")

print(parts)
print(parts[1])   # is this the whole address?

What went wrong?

`.split(",")` cuts at **every comma**. But your address also contains commas, so one address gets broken into several pieces.

To handle this properly using only Python's **standard library**, use the `csv` module. It knows that commas separate columns, but commas inside quotes belong to the same value.

```python
import csv

with open(path, newline="") as f:
    rows = list(csv.reader(f))
```

Now:

* `rows` is a list of rows
* each row is a list
* each item in a row is one column


In [ ]:
import csv

with open(path, "r", newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

print(rows[0])    # the header, as a list
print(rows[1])    # the first place, as a list
print(len(rows))  # number of rows, header included

### Picking one value

`rows[1]` is the first place. `rows[1][0]` is its first value, the name. Column positions follow the header:

| position | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|---|
| column | `name` | `address` | `lat` | `lon` | `category` | `notes` | `rating` | `accessibility` |

Python counts from `0`, and `-1` means the last one, so `rows[-1]` is your last place.

One catch: the reader returns **every** value as a string, including the numbers. `float()` turns a number that is stored as text into a real number.

In [ ]:
print(rows[1][0])                       # name
print(rows[1][4])                       # category

print(rows[1][2], type(rows[1][2]))     # latitude, but as text
latitude = float(rows[1][2])
print(latitude, type(latitude))         # now a number

### ✏️ Try it out!

Using `rows` and indexing:

1. Print the name and category of the **last** place in your file.
2. Store its longitude as a number in `longitude`, and print `type(longitude)`.
3. Print `len(rows)`. How many places is that, given that one row is the header?

**Explain to your partner:** why does `float()` need to be there?

In [ ]:
# Your code here.

<details>
<summary>Check your approach</summary>

```python
print(rows[-1][0], rows[-1][4])
longitude = float(rows[-1][3])
print(type(longitude))
print(len(rows))
```

</details>

### Check your ratings

In Lab 1.2, the `rating` column (position `6`) was optional, so your file may have a rating for every place, for some, or for none. Click your CSV once in the file browser to see it in the table viewer and look at that column.

Rate any place that does not have one yet:

1. Right-click your CSV and choose **Open With, Editor**, as you did in Lab 1.2.
2. In the `rating` column, type `love it`, `like it`, or `just okay`, lowercase. Give at least one place `love it`.
3. Save the file.

It is fine to leave one or two places blank. The sections below look for missing values, and a real dataset always has some.

Python read the file before you changed it, so read it again. The cell below is the same as before; it just refreshes `rows`.

In [ ]:
with open(path, "r", newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

print(rows[1][0], "->", rows[1][6])   # name and rating of the first place

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 2. Conditionals: is there a rating, and is it a favourite?

These are two different questions.

- `bool(rating)` asks: is there anything in the string? An empty string `""` is `False`; any other string is `True`.
- `rating == "love it"` asks: is the string exactly this text?

Run the cell. Then change `rating` to `"love it"`, and then to `""`, running it each time.

In [ ]:
rating = "just okay"
print("Has a rating:", bool(rating))
print("Is a favourite:", rating == "love it")

### One place, one decision

`if` / `elif` / `else` picks one action out of several. `not` flips a Boolean, so `not rating` is `True` exactly when the rating is empty.

Read the three branches before running the cell. Which message will appear for your first place?

In [ ]:
i = 1  # `i` is by convention used to show index in python, here 1 is the index of the first place in the list of rows
name = rows[i][0]
rating = rows[i][6]
# Check for a missing rating before interpreting its value.
if not rating:
    print(name, "has no rating yet.")
elif rating == "love it":
    print(name, "is a favourite.")
else:
    print(name, "has a rating, but is not a favourite.")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 1. Read your places file

In **Lab 1.2**, you created a CSV file called something like:

```text
firstname_studentid_myplaces.csv
```

Today, we are going to read that file with Python.

### First: where is the file?

Your CSV is in your **lab folder**, but this notebook is in a different folder. Python therefore needs directions to the file, a.k.a. a **path**.

You have already seen paths such as:

```text
data/raw/file.csv
```

That means: starting from where we are now, go into `data`, then `raw`, then find `file.csv`.

Sometimes the file is not inside the current folder. Then we use:

```text
..
```

`..` means **go up one folder**.

For this notebook, your CSV should be at:

```text
../../../lab/w02/firstname_studentid_myplaces.csv
```

The `../../..` part means **go up three folders**, and then `lab/w02/` tells Python where to go next.

### Find your file

In the DataHub file browser:

1. Open `materials-fa26/lab/w02/`
2. Find your `myplaces.csv` file.
3. Replace `firstname_studentid_myplaces.csv` below with **your actual filename**.

Last week, we also learned that Python comes with a **standard library**: useful modules that are already installed with Python.

`os` is one of those modules. Here, we will use it for one simple task: **checking whether Python can find your file.**

```python
import os

path = "../../../lab/w02/firstname_studentid_myplaces.csv"

os.path.exists(path)
```

At first, this will probably return:

```text
False
```

That is because `firstname_studentid_myplaces.csv` is only a placeholder.

Change it to your actual filename and run the cell again.

```text
True
```

means Python found the file.

**Key idea:** A path is simply the directions Python follows to find a file.

`is_favourite` is a Boolean: `True` or `False`. Replacing an item changes its value, not the list's length, and `len(place_ratings)` confirmed it. Recalculate `is_favourite` after editing the rating: an earlier assignment does not update on its own.



### ✏️ Try it out!

Copy the decision code into the cell below and change `i` so that it points at a place you rated `love it`, then at one you rated `like it` or `just okay`, then at one without a rating. Run it each time. Do all three branches work?

If every place in your file has a rating, test the third branch by changing the line `rating = rows[i][6]` to `rating = ""`.

**Explain to your partner:** why would `if rating:` alone be the wrong test for a favourite?

In [ ]:
# Copy the decision code here, then change i to test each branch.

**Pause here for the lesson on loops.**

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 3. Loops: repeat for every place

A `for` loop repeats an action for each item in a list. `rows` is a list, so this prints the rating of every row.

Run the cell. The first line printed is `rating`: the header is a row too.

In [ ]:
for row in rows:
    print(row[6])

To skip the header, loop over positions instead of rows, and start at `1`. `range(1, len(rows))` gives the positions `1`, `2`, `3`, ... up to the last row, and `i` takes the next one each time the loop runs.

In [ ]:
for i in range(1, len(rows)):
    print(rows[i][0], "->", rows[i][6])

### ✏️ Try it out!

Write a loop that prints each place's **name, category, and rating** on one line, without the header.

In [ ]:
# Your code here.

<details>
<summary>Check your approach</summary>

```python
for i in range(1, len(rows)):
    print(rows[i][0], "|", rows[i][4], "|", rows[i][6])
```

</details>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 4. Loop + conditional: which places would you recommend?

Now combine the two. To collect the places rated `"love it"`:

1. Start with an empty list.
2. Visit every position with a loop.
3. Inside the loop, use `if` to append the name only when the rating matches.

`.append()` adds one item to the end of a list.

In [ ]:
favourites = []
for i in range(1, len(rows)):
    if rows[i][6] == "love it":
        favourites.append(rows[i][0])

print("Favourite places:", favourites)

### ✏️ Try it out!

Pick a category that occurs in your file and store it in `chosen_category`. Build `recommendations`: the names in that category **and** rated `"love it"`. Use `and` to combine the two comparisons, then print the result.

An empty result can be correct. Check it against your file.

In [ ]:
# Set chosen_category, then build and print recommendations.

<details>
<summary>Check your approach</summary>

```python
chosen_category = "food"
recommendations = []
for i in range(1, len(rows)):
    if rows[i][4] == chosen_category and rows[i][6] == "love it":
        recommendations.append(rows[i][0])
print(recommendations)
```

</details>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## 5. Counting instead of collecting

Sometimes you only need a number. Start a counter at `0`, and add `1` each time the condition is met.

In [ ]:
loved_count = 0
for i in range(1, len(rows)):
    if rows[i][6] == "love it":
        loved_count = loved_count + 1

print("Number of favourites:", loved_count)
print("Names collected:", len(favourites))

### ✏️ Try it out!

Count the places **without** a rating and print the count. Use `not rows[i][6]` as the condition. The answer should match the number of blanks in your file; `0` is correct if you rated everything.

In [ ]:
# Start missing_count at zero, then loop through the rows.

<details>
<summary>Check your approach</summary>

```python
missing_count = 0
for i in range(1, len(rows)):
    if not rows[i][6]:
        missing_count = missing_count + 1
print("Missing ratings:", missing_count)
```

</details>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## Before you finish

1. If any rating is still blank, open your CSV in the editor, fill it in, and save. Rerun the cell that refreshes `rows` and the cells from section 3 on. The missing count should now be `0`.
2. **Keep this notebook. There is nothing to submit today.** Class activities are for you: each session builds on the one before, and later in the course you will submit the maps and other results derived from your places. Keep this notebook and your CSV where you can find them on DataHub.

**Check with your partner:** what is the difference between `if rating:` and `if rating == "love it":`? Point to one place where your code uses a loop and a conditional together.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## Optional: see the filter

The chart code is provided. Before you run it, predict which categories will contain a `"love it"` place. Each category gets up to two bars: places kept by the filter and places filtered out.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# One category and one filter result per place, in file order.
categories = []
kept = []
for i in range(1, len(rows)):
    categories.append(rows[i][4])
    if rows[i][6] == "love it":
        kept.append("kept")
    else:
        kept.append("filtered out")

sns.set_theme(style="whitegrid")
ax = sns.countplot(x=categories, hue=kept, hue_order=["kept", "filtered out"])
ax.set_title("Which places are rated 'love it'?")
ax.set_xlabel("Category")
ax.set_ylabel("Number of places")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />


## Summary

| Question | Python |
|---|---|
| Where is my file? | its path; `os.path.exists(path)` says whether Python can see it |
| How do I open a file? | `with open(path) as f:` |
| How do I read a CSV? | `import csv`, then `csv.reader(f)` |
| Why is the latitude a string? | files hold text; `float()` makes it a number |
| Which value is this? | `rows[i][6]`: row `i`, column `6`; `rows[-1]` is the last row |
| Is there a rating? | `if rating:`, or `if not rating:` for the missing ones |
| Is it exactly `"love it"`? | `rating == "love it"` |
| Which action should run? | `if` / `elif` / `else` |
| How do I visit every row? | `for row in rows:` |
| How do I skip the header? | `for i in range(1, len(rows)):` |
| How do I collect matches? | an empty list, a loop, an `if`, and `.append()` |
| How do I count matches? | start at `0` and add `1` for each match |

Wednesday: the rule `== "love it"` gets a name of its own, a function.